# Task 1 — Reproduce Adam by Hand

> **Ask.** Take one weight and five gradients, compute $m$, $v$, $\hat m$, $\hat v$
> and the resulting step yourself, then check each against PyTorch. They should
> agree to several decimal places.

This notebook does three things:

1. Implements the Adam update in plain Python, printing **every** intermediate
   quantity for all five steps.
2. Drives `torch.optim.Adam` / `torch.optim.AdamW` with the *same* weight and the
   *same* five gradients and diffs the two, number by number.
3. Repeats the check with **decoupled weight decay** turned on, so we can see
   exactly where AdamW departs from Adam.

### The update rule (what we are reproducing)

For step $t$, gradient $g_t$, hyper-parameters $\alpha$ (lr), $\beta_1,\beta_2,\epsilon$:

$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1)\, g_t \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2)\, g_t^2 \\
\hat m_t &= m_t / (1-\beta_1^{\,t}) \\
\hat v_t &= v_t / (1-\beta_2^{\,t}) \\
\Delta_t &= \alpha \,\hat m_t / (\sqrt{\hat v_t} + \epsilon) \\
w_t &= w_{t-1} - \Delta_t \qquad\text{(Adam)} \\
w_t &= w_{t-1} - \Delta_t - \alpha\,\lambda\, w_{t-1} \qquad\text{(AdamW, decoupled decay }\lambda)
\end{aligned}
$$

In [ ]:
import sys, os, math, time, json, warnings
sys.path.insert(0, os.path.abspath("../src"))
warnings.filterwarnings("ignore")

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from s11.utils import set_seed, get_device, savefig, plot_style, AIM_REPO
plot_style()
set_seed(1337)
DEVICE = get_device()
torch.set_float32_matmul_precision("high")
print(f"torch {torch.__version__} | device = {DEVICE} "
      f"| {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'cpu'}")
print(f"Aim repo: {AIM_REPO}  (browse it later with:  uv run aim up)")

## 1. The setup: one weight, five gradients

We pick a single scalar weight and five hand-chosen gradients (mixed signs and
magnitudes, so nothing is accidentally symmetric). Hyper-parameters are the Adam
defaults **except** `beta2`, where we deliberately show both the classic
`0.999` and the LLM-style `0.95` later on.

In [ ]:
from s11.optim import run_adam_by_hand, AdamState, adam_step

w0    = 0.7213                       # the one weight
grads = [0.15, -0.30, 0.05, 0.22, -0.11]   # five gradients

LR, B1, B2, EPS = 1e-3, 0.9, 0.999, 1e-8

print(f"w0 = {w0}")
print(f"gradients = {grads}")
print(f"lr={LR}  beta1={B1}  beta2={B2}  eps={EPS}\n")

trace = run_adam_by_hand(w0, grads, lr=LR, beta1=B1, beta2=B2, eps=EPS)

df = pd.DataFrame(trace)[["t", "m", "v", "m_hat", "v_hat", "step", "param_before", "param_after"]]
pd.set_option("display.float_format", lambda x: f"{x:.10f}")
print("HAND-COMPUTED ADAM TRACE")
print(df.to_string(index=False))

### Step-by-step arithmetic for step 1 (spelled out)

To make the table above auditable, here is step 1 done symbol-by-symbol.

In [ ]:
g1 = grads[0]
m1 = B1*0.0 + (1-B1)*g1
v1 = B2*0.0 + (1-B2)*g1**2
m1_hat = m1 / (1 - B1**1)
v1_hat = v1 / (1 - B2**1)
step1  = LR * m1_hat / (math.sqrt(v1_hat) + EPS)

print(f"g1                 = {g1}")
print(f"m1 = (1-b1)*g1     = {m1:.10f}")
print(f"v1 = (1-b2)*g1^2   = {v1:.12f}")
print(f"1-b1^1             = {1-B1**1:.10f}")
print(f"1-b2^1             = {1-B2**1:.10f}")
print(f"m1_hat             = {m1_hat:.10f}   (== g1, since 1-b1^1 = 1-b1)")
print(f"v1_hat             = {v1_hat:.12f}   (== g1^2)")
print(f"sqrt(v1_hat)       = {math.sqrt(v1_hat):.10f}   (== |g1|)")
print(f"step1 = lr*m1_hat/(sqrt(v1_hat)+eps) = {step1:.10f}")
print(f"      ~ lr * sign(g1) = {LR*np.sign(g1):.10f}   <- Adam's 'unit step' behaviour on step 1")
print(f"w1 = w0 - step1    = {w0 - step1:.10f}")

## 2. Check against PyTorch

We create a leaf tensor initialised to `w0`, and for each of the five steps we
**write the gradient by hand** into `p.grad` and call `opt.step()`. No autograd,
no loss — just the exact same five numbers fed to the reference optimizer.

In [ ]:
def torch_adam_trace(w0, grads, lr, b1, b2, eps, weight_decay=0.0, kind="adam"):
    p = torch.tensor([w0], dtype=torch.float64, requires_grad=True)
    Opt = torch.optim.AdamW if kind == "adamw" else torch.optim.Adam
    opt = Opt([p], lr=lr, betas=(b1, b2), eps=eps, weight_decay=weight_decay)
    rows = []
    for t, g in enumerate(grads, start=1):
        opt.zero_grad()
        p.grad = torch.tensor([g], dtype=torch.float64)
        before = p.detach().clone()
        opt.step()
        st = opt.state[p]
        rows.append(dict(
            t=t,
            m=st["exp_avg"].item(),
            v=st["exp_avg_sq"].item(),
            m_hat=st["exp_avg"].item() / (1 - b1**t),
            v_hat=st["exp_avg_sq"].item() / (1 - b2**t),
            step=(before - p.detach()).item(),
            param_before=before.item(),
            param_after=p.detach().item(),
        ))
    return pd.DataFrame(rows)

tdf = torch_adam_trace(w0, grads, LR, B1, B2, EPS, kind="adam")
print("PYTORCH torch.optim.Adam TRACE")
print(tdf.to_string(index=False))

In [ ]:
# Numeric diff, hand vs torch, for every quantity at every step
cols = ["m", "v", "m_hat", "v_hat", "step", "param_after"]
diff = (df[cols].reset_index(drop=True) - tdf[cols].reset_index(drop=True)).abs()
diff.insert(0, "t", df["t"].values)
print("ABSOLUTE DIFFERENCE  |hand - pytorch|\n")
print(diff.to_string(index=False, float_format=lambda x: f"{x:.2e}"))
print(f"\nmax abs diff over the whole trace = {diff[cols].to_numpy().max():.3e}")
assert diff[cols].to_numpy().max() < 1e-12, "hand and torch disagree!"
print("PASS  — hand-rolled Adam matches torch.optim.Adam to < 1e-12 (float64)")

## 3. Where AdamW differs: decoupled weight decay

Turn on weight decay $\lambda = 0.1$. Classic **Adam** (`torch.optim.Adam` with
`weight_decay`) folds $\lambda w$ *into the gradient* before the moments are
updated — so the decay is itself smoothed by $m$ and rescaled by $\sqrt{\hat v}$.
**AdamW** applies $-\alpha\lambda w$ *after* the Adam step, untouched by the
moment estimates. We reproduce both.

In [ ]:
WD = 0.1

hand_adam_l2 = run_adam_by_hand(w0, grads, lr=LR, beta1=B1, beta2=B2, eps=EPS,
                                weight_decay=WD, decoupled=False)
hand_adamw   = run_adam_by_hand(w0, grads, lr=LR, beta1=B1, beta2=B2, eps=EPS,
                                weight_decay=WD, decoupled=True)

t_adam_l2 = torch_adam_trace(w0, grads, LR, B1, B2, EPS, weight_decay=WD, kind="adam")
t_adamw   = torch_adam_trace(w0, grads, LR, B1, B2, EPS, weight_decay=WD, kind="adamw")

# NOTE: compare the *observable* — m, v and the resulting weight. (Our hand trace's
# "step" column is the pure Adam step; for AdamW the decay is applied separately to
# param_after, so the two "step" columns are not like-for-like — param_after is.)
def maxdiff(trace, tdf):
    a = pd.DataFrame(trace)[["m","v","param_after"]].reset_index(drop=True)
    b = tdf[["m","v","param_after"]].reset_index(drop=True)
    return (a - b).abs().to_numpy().max()

d_l2, d_w = maxdiff(hand_adam_l2, t_adam_l2), maxdiff(hand_adamw, t_adamw)
print(f"Adam + L2 decay   : max |hand - torch|  (m, v, weight) = {d_l2:.3e}")
print(f"AdamW (decoupled) : max |hand - torch|  (m, v, weight) = {d_w:.3e}")
assert max(d_l2, d_w) < 1e-12, "decay variants disagree!"
print("PASS — both weight-decay variants match torch to < 1e-12 (float64)")

comp = pd.DataFrame({
    "t": range(1, 6),
    "w  (Adam, no decay)":  [r["param_after"] for r in trace],
    "w  (Adam + L2)":       [r["param_after"] for r in hand_adam_l2],
    "w  (AdamW decoupled)": [r["param_after"] for r in hand_adamw],
})
print()
print(comp.to_string(index=False, float_format=lambda x: f"{x:.8f}"))

## 4. Charts

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
t = [r["t"] for r in trace]

ax[0,0].plot(t, [r["m"] for r in trace], "o-", label="m (biased)")
ax[0,0].plot(t, [r["m_hat"] for r in trace], "s--", label=r"$\hat m$ (corrected)")
ax[0,0].set_title("First moment"); ax[0,0].set_xlabel("step"); ax[0,0].legend()

ax[0,1].plot(t, [r["v"] for r in trace], "o-", label="v (biased)")
ax[0,1].plot(t, [r["v_hat"] for r in trace], "s--", label=r"$\hat v$ (corrected)")
ax[0,1].set_title("Second moment"); ax[0,1].set_xlabel("step"); ax[0,1].legend()

ax[0,2].plot(t, [r["step"] for r in trace], "o-", color="crimson")
ax[0,2].axhline(LR, ls=":", color="grey", label="lr")
ax[0,2].axhline(-LR, ls=":", color="grey")
ax[0,2].set_title(r"Update $\Delta_t$"); ax[0,2].set_xlabel("step"); ax[0,2].legend()

ax[1,0].plot(t, [1-B1**i for i in t], "o-", label=r"$1-\beta_1^t$")
ax[1,0].plot(t, [1-B2**i for i in t], "s-", label=r"$1-\beta_2^t$")
ax[1,0].set_title("Bias-correction denominators"); ax[1,0].set_xlabel("step"); ax[1,0].legend()

ax[1,1].plot(t, [r["param_after"] for r in trace], "o-", label="hand")
ax[1,1].plot(tdf["t"], tdf["param_after"], "x", ms=12, label="pytorch")
ax[1,1].set_title("Weight trajectory: hand vs torch"); ax[1,1].set_xlabel("step"); ax[1,1].legend()

ax[1,2].plot(t, [r["param_after"] for r in trace], "o-", label="Adam")
ax[1,2].plot(t, [r["param_after"] for r in hand_adam_l2], "s-", label="Adam+L2")
ax[1,2].plot(t, [r["param_after"] for r in hand_adamw], "^-", label="AdamW")
ax[1,2].set_title("Weight decay variants"); ax[1,2].set_xlabel("step"); ax[1,2].legend()

plt.tight_layout()
savefig(fig, "t1_adam_by_hand.png")
plt.show()

## 5. Findings

| quantity | agreement (hand vs `torch.optim`) |
|---|---|
| $m,\;v,\;\hat m,\;\hat v$, update, weight | `< 1e-12` in float64 (all five steps) |
| Adam + L2 weight decay | `< 1e-12` |
| AdamW decoupled decay | `< 1e-12` |

**What the numbers show**

- On **step 1** the update is almost exactly $\alpha\cdot\mathrm{sign}(g_1)$: the
  bias-correction factors cancel ($\hat m_1 = g_1$, $\sqrt{\hat v_1}=|g_1|$), so
  Adam takes a *unit* step regardless of gradient magnitude.
- The biased moments $m,v$ start near zero and crawl upward; the corrected
  $\hat m,\hat v$ are what actually drive the step, and for the first few steps
  they are **much** larger than the biased versions (dividing by $1-\beta_2^t\approx
  10^{-3}$ on step 1).
- **Adam + L2** vs **AdamW**: with $\lambda=0.1$ the trajectories separate by step
  2–3. In L2-Adam the decay term gets divided by $\sqrt{\hat v}$ (so parameters
  with large gradients are decayed *less*); AdamW decays every weight by the same
  multiplicative factor $(1-\alpha\lambda)$. This decoupling is the entire point
  of AdamW.

> **Myth check.** The lecture transcript says *"bias correction is what AdamW is
> doing."* That is not quite right. **Bias correction** (dividing by $1-\beta^t$)
> fixes the zero-initialisation bias of the EMAs and is part of *plain Adam*.
> **AdamW's** contribution is *decoupled weight decay*. They are independent
> features — Task 2 pulls bias correction out on its own.